# Type Coercion

When Pydantic deserializes data, one of the things it does is perform validation. This includes ensuring that the model instance data ends up as the correct type.

Let's look at an example:

In [27]:
from pydantic import BaseModel, ValidationError

In [28]:
class Coordinates(BaseModel):
    x: float
    y: float

Notice how we are insisting that `x` and `y` should be floats.

Let's deserialize some data:

In [29]:
p1 = Coordinates(x=1.1, y=2.2)
p1

Coordinates(x=1.1, y=2.2)

We can see our field definitions:

In [30]:
Coordinates.model_fields

{'x': FieldInfo(annotation=float, required=True),
 'y': FieldInfo(annotation=float, required=True)}

As you can see, the defined type for each field is `float`. 

And indeed, if we check the type of the fields, we get the expected result:

In [31]:
type(p1.x)

float

But what happens if the data we provide for deserialization is not an exact type match?

Pydantic will attempt to "transform" the data into the correct type - this is called type **coercion**

Let's see this:

In [32]:
p2 = Coordinates(x=0, y="1.2")
p2

Coordinates(x=0.0, y=1.2)

As you can see, Pydantic was able to coerce the **integer** `0`, and the **string** `"1.2"` to a float value:

In [33]:
type(p2.x), type(p2.y)

(float, float)

Pydantic is not always able to perform the type coercion. In fact, we can even choose the level of type coercion that we find acceptable.

By default, the type coercion is termed **lax** - and it attempts a variety of type coercions.

But, as we'll see later, we have the option to change that, to a **strict** mode that is far less forgiving when incorrect data types are provided in the data.

Pydantic docs that describes what type coercions will be attempted in either of these modes, is located here:

[https://docs.pydantic.dev/latest/concepts/conversion_table/](https://docs.pydantic.dev/latest/concepts/conversion_table/)

If you look at that conversion table, you'll notice, for example, that in lax mode, and dealing with Python data types, input data that is float, int, or Decimal will be coerced to a float. However, strings will be coerced to floats only under certain conditions.

In strict mode, notice that string to float conversion is not supported (so it will raise a validation error).

Use this table when considering type coercion because things are not always "obvious".

For example, with this model:

In [34]:
class Model(BaseModel):
    field: str

We know that all objects in Python have a `str()` representation, so we might expect to be able to pass any type for `field` and have Pydantic coerce it to a string.

But that is not the case:

In [35]:
try:
    Model(field=100)
except ValidationError as ex:
    print(ex)

1 validation error for Model
field
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


And this is probably a good thing as allowing this could lead to unintended problems where we deserialize and object to a string when we never intended for that to happen (our source data maybe changed on us - and auto coercing to a string would hide a potential bug).

Here's what I mean:

We are querying a REST API and getting some JSON back from that API, which we model this way:

<div style="
    direction: rtl;
    background:  #282b2b;
    padding: 40px 30px;
    border-radius: 14px;
    font-family: 'Vazirmatn', Tahoma;
    line-height: 2;
    color: #fcfcfc;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 20px;
    width: 85%; margin: 10px auto;
">

<span style="font-size: 38px; margin-right:25px">🔍</span>
تحلیل: اینجا یک مدل ساده داریم که یک فیلد email از نوع رشته (str) می‌گیرد. API هم دقیقاً همان چیزی را می‌فرستد که انتظار داریم (یک ایمیل متنی ساده). همه چیز عالی کار می‌کند.

</div>


In [36]:
class Contact(BaseModel):
    email: str

In [37]:
initial_json_data = '''
{
    "email": "inewton@principia.com"
}
'''

This deserializes just fine:

In [38]:
Contact.model_validate_json(initial_json_data)

Contact(email='inewton@principia.com')

But now, suppose that API changes it's response model, and we are unaware of the change.

The response data now looks like this:

<div style="
    direction: rtl;
    background:  #f505c1;
    padding: 40px 30px;
    border-radius: 14px;
    font-family: 'Vazirmatn', Tahoma;
    line-height: 2;
    color: #fcfcfc;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 20px;
    width: 85%; margin: 10px auto;
">

<span style="font-size: 38px; margin-right:25px">🔍</span>
<strong>قدم دوم: تغییر ناگهانی API (کابوس برنامه‌نویس‌ها)</strong>
تحلیل: به هر دلیلی (مثلاً آپدیت شدن سرور طرف مقابل)، ساختار داده تغییر کرده است. حالا email دیگر یک رشته ساده نیست، بلکه یک “دیکشنری تو در تو” است که شامل ایمیل شخصی و کاری می‌شود. ما هم روحمان از این تغییر خبر ندارد!
</div>


In [ ]:
new_json_data = '''
{
    "email": {
        "personal": "inewton@principia.com",
        "work": "isaac.newton@themint.com"
    }
}
'''

Tring to deserialize this data will not work, and we'll immediately know something is wrong with our app (and we can then go in and fix it):

In [39]:
try:
    Contact.model_validate_json(new_json_data)
except ValidationError as ex:
    print(ex)

1 validation error for Contact
email
  Input should be a valid string [type=string_type, input_value={'personal': 'inewton@pri...aac.newton@themint.com'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


<div style="
    direction: rtl;
    background:  #4f9e53;
    padding: 40px 30px;
    border-radius: 14px;
    font-family: 'Vazirmatn', Tahoma;
    line-height: 2;
    color: #fcfcfc;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 20px;
    width: 85%; margin: 10px auto;
">

<span style="font-size: 38px; margin-right:25px">🔍</span>
تحلیل: وقتی پایدانتیک سعی می‌کند این داده جدید را بخواند، بلافاصله ارور می‌دهد و می‌گوید: “من منتظر یک رشته بودم، اما تو به من یک دیکشنری دادی!”
نکته  “این ارور دادن، یک موهبت است! پایدانتیک در همان نقطه ورود (Entry Point)، جلوی داده کثیف را گرفت تا وارد سیستم شما نشود.”
</div>


What if Pydantic had instead just decided to deserialize that `email` complex object (a nested dictionary basically) into it's string representation?

We can mimic the behavior this way:

In [40]:
new_data = {
    "email": {
        "personal": "inewton@principia.com",
        "work": "isaac.newton@themint.com"
    }
}

In [41]:
Contact(email=str(new_data['email']))

Contact(email="{'personal': 'inewton@principia.com', 'work': 'isaac.newton@themint.com'}")

And you can see that we now have a string representation of the email dictionary - no exceptions, and our code is likely to break from that point forward, depending on how we use the `email` field.

<div dir="rtl" align="right" style="width: 85%; margin: 10px auto; background-color: #fff8dc; color: #8b6508; padding: 20px; border-radius: 8px; border-right: 5px solid #daa520; line-height: 1.8;"

### دیدگاه اول: Type Coercion (تبدیل خودکار/ضمنی)
در این حالت، زبان برنامه‌نویسی یا فریم‌ورک (مثل Pydantic در FastAPI) پشت صحنه و بدون دخالت تو، یک نوع داده را به نوع دیگری تبدیل می‌کند تا کار راه بیفتد.

* **مثال ساده:** فرض کن در یک محاسبه ریاضی داریم $x=10$ (عدد صحیح) و $y=5.5$ (اعشاری). وقتی می‌نویسی $x+y$، پایتون به صورت خودکار $x$ را به اعشاری تبدیل می‌کند تا جواب $15.5$ شود. این یک Coercion است.
* **مثال در Pydantic (با فرض نصب توسط `uv`):** کاربر در فرم HTML سن خود را می‌نویسد. دیتایی که به سمت FastAPI می‌آید یک رشته است (مثلاً `"25"`). اما تو در مدل نوشته‌ای `age: int`. پایدانتیک به طور خودکار `"25"` را به عدد $25$ تبدیل می‌کند. این یعنی **Pydantic Type Coercion**.

### دیدگاه دوم: Type Casting (تبدیل دستی/صریح)
اینجا تو به عنوان برنامه‌نویس، صراحتاً و با دستورات کدنویسی، نوع داده را عوض می‌کنی. هیچ چیز خودکاری در کار نیست.

* **مثال ساده:** تو خودت دست به کار می‌شوی و می‌نویسی `int("25")` تا رشته را به عدد تبدیل کنی. این کَستینگ (Casting) است.

> «بچه‌ها، **Casting** مثل این است که خودتان یک دلار را ببرید صرافی و صراحتاً بگویید این را به تومان تبدیل کن. اما **Coercion** مثل این است که در یک فروشگاه در تهران، یک اسکناس یک دلاری روی میز بگذارید و فروشنده خودش در ذهنش آن را به تومان تبدیل کند و بقیه پولتان را بدهد! پایدانتیک در حالت پیش‌فرض (Lax Mode) دقیقاً نقش همان فروشنده را بازی می‌کند.»

پس وقتی مستندات Pydantic می‌گوید "Pydantic tries to coerce it"، یعنی "پایدانتیک سعی می‌کند خودش زیرسبیلی و به صورت خودکار، داده را به چیزی که شما خواسته‌اید تبدیل کند".

</div>


<div dir="rtl" align="right" style="width: 85%; margin: 10px auto; background-color: #fff8dc; color: #8b6508; padding: 20px; border-radius: 8px; border-right: 5px solid #daa520; line-height: 1.8;"

### مفهوم اصلی: Lax Mode vs Strict Mode

پایدانتیک دو حالت برای اعتبارسنجی دارد:

1. **Lax (حالت منعطف - پیش‌فرض)**:  
   پایدانتیک سعی می‌کند با “سخت‌گیری کم”، داده‌ها را به تایپ مورد نظر شما **تبدیل (Coerce)** کند.  
   مثلا رشته `"123"` را به عدد `123` تبدیل می‌کند.

2. **Strict (حالت سخت‌گیرانه)**:  
   فقط در صورتی داده را قبول می‌کند که **دقیقاً همان تایپ** باشد  
   یا در ستون Strict آن جدول، تیک خورده باشد.

</div>


![alt text](<Screenshot 2026-04-13 174438.png>)

![alt text](<Screenshot 2026-04-13 174516.png>)

<div style="direction: rtl; text-align: right;width: 85%; margin: 10px auto; font-family: 'Vazirmatn', Tahoma; line-height: 1.9;">

### 🧩 تحلیل ستون‌های جدول برای آموزش:

**Field Type:**  
تایپی که شما در مدل تعریف کردید (مثلاً `bool`).

**Input:**  
چیزی که کاربر فرستاده (مثلاً رشته `"on"`).

**Strict:**  
اگر تیک داشته باشد، یعنی حتی در حالت سخت‌گیرانه هم این **تبدیل (Type Coercion)** انجام می‌شود.  
اگر تیک نداشته باشد، فقط در حالت معمولی (**Lax**) کار می‌کند.

**Conditions:**  
شرایط خاص (مثلاً برای Boolean، فقط رشته‌های خاصی مثل `'yes'`, `'true'`, `'1'` قبول هستند).

</div>


<div style="direction: rtl;width: 85%; margin: 10px auto; text-align: right; font-family: 'Vazirmatn', Tahoma; line-height: 1.9;">

### ⚙️ نکته فنی مهم

در FastAPI، پایدانتیک به‌صورت پیش‌فرض در حالت **Lax** عمل می‌کند تا کار با فرانت‌اند و فرم‌های HTML راحت‌تر باشد.  
اما در سیستم‌های **مالی، بانکی یا هر جایی که داده حساس است** حتماً باید از ستون **Strict** در جدول استفاده کنی و مدل‌های خود را محدود و دقیق تعریف کنی.

</div>


<div style="
    direction: rtl;
    background:  #282b2b;
    padding: 40px 30px;
    border-radius: 14px;
    font-family: 'Vazirmatn', Tahoma;
    line-height: 2;
    color: #fcfcfc;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 20px;
    width: 85%; margin: 10px auto;
">

<span style="font-size: 38px; margin-right:25px">🔍</span>
<strong>چند مثال برای روشن شدن توضیحات بالا</strong>

</div>


<div style="width: 85%; margin: 10px auto; direction: rtl; text-align: right; font-family: 'Vazirmatn', Tahoma; line-height: 2;">

<strong>راهکار اول: استفاده از ConfigDict (سخت‌گیری در سطح کل مدل)</strong>  
این روش زمانی استفاده می‌شود که می‌خواهی کل API تو نسبت به ورودی‌ها حساس باشد و اجازه تبدیل‌های خودکار (مثل تبدیل رشته به بولین) را ندهد.

</div>


In [42]:
from pydantic import BaseModel, ConfigDict, ValidationError

class UserSettings(BaseModel):
    #Strict تنظیم مدل روی حالت 
    model_config = ConfigDict(strict=True)
    
    is_active: bool

# مثال ۱: ورودی صحیح
try:
    user = UserSettings(is_active=True)
    print("✅ Success:", user.is_active)
except ValidationError as e:
    print("❌ Error:", e.json())

#قبول میشد دراینجاخطامیدهدLax ورودی که در حالت 
try:
    #   .تیک ندارد:boolبرای strict :در حالت "yes" بر اساس جدول رشته 
    user = UserSettings(is_active="yes") 
except ValidationError as e:
    print("❌ Error: 'yes' is not allowed in strict mode!",e)


✅ Success: True
❌ Error: 'yes' is not allowed in strict mode! 1 validation error for UserSettings
is_active
  Input should be a valid boolean [type=bool_type, input_value='yes', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/bool_type


In [43]:
print(UserSettings.model_config)

{'strict': True}


<div style="width: 85%; margin: 10px auto; direction: rtl; text-align: right; font-family: 'Vazirmatn', Tahoma; line-height: 2;">

<strong>راهکار دوم: استفاده از فیلد اختصاصی (سخت‌گیری در سطح فیلد)</strong>  
گاهی می‌خواهی فقط برای یک فیلد خاص (مثل قیمت یا تاریخ) سخت‌گیر باشی و بقیه فیلدها منعطف بمانند.

</div>


In [44]:
from pydantic import BaseModel, Field
from datetime import date

class Transaction(BaseModel):
    # این فیلد فقط تایپ دقیق date یا رشته استاندارد را در حالت Strict می‌پذیرد
    due_date: date = Field(strict=True)
# این فیلد در حالت پیش‌فرض (Lax) است
    description: str     

# طبق جدول، اگر float بفرستیم (timestamp)، در حالت Strict خطا می‌دهد
try:
    t = Transaction(due_date=1712995200.0, description="Payment")
except ValidationError as e:
    print("❌ Date cannot be a float in strict mode!")


❌ Date cannot be a float in strict mode!


In [45]:
from pydantic import BaseModel

class Test(BaseModel):
    x: int

print(Test.model_config)

{}
